In [1]:
import timm

In [2]:
from google.colab import drive
import os
import pandas as pd
from sklearn.model_selection import train_test_split

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
DATASET_DIR = "/content/flickr30k"
IMAGE_DIR = f"{DATASET_DIR}/Images"
CAPTION_FILE = f"{DATASET_DIR}/captions.txt"

In [4]:
# Copying 8k to Colabs local SSD
if not os.path.exists("/content/flickr30k.zip"):
  !cp -r "/content/drive/MyDrive/MMRetrieval/flickr30k.zip" /content/

In [5]:
!unzip "/content/flickr30k.zip" -d "/content/flickr30k"

Streaming output truncated to the last 5000 lines.
  inflating: /content/flickr30k/Images/2410320522.jpg  
  inflating: /content/flickr30k/Images/2404747797.jpg  
  inflating: /content/flickr30k/Images/241046599.jpg  
  inflating: /content/flickr30k/Images/241345639.jpg  
  inflating: /content/flickr30k/Images/2409312675.jpg  
  inflating: /content/flickr30k/Images/241346471.jpg  
  inflating: /content/flickr30k/Images/2404959574.jpg  
  inflating: /content/flickr30k/Images/2410399168.jpg  
  inflating: /content/flickr30k/Images/2406591500.jpg  
  inflating: /content/flickr30k/Images/2413495734.jpg  
  inflating: /content/flickr30k/Images/241346794.jpg  
  inflating: /content/flickr30k/Images/2404488732.jpg  
  inflating: /content/flickr30k/Images/241347664.jpg  
  inflating: /content/flickr30k/Images/241345522.jpg  
  inflating: /content/flickr30k/Images/241347300.jpg  
  inflating: /content/flickr30k/Images/241347496.jpg  
  inflating: /content/flickr30k/Images/2407214681.jpg  
  inf

In [6]:
print("Images:", len(os.listdir(IMAGE_DIR)))

df = pd.read_csv(CAPTION_FILE)

print(df.head())
print(df.columns)
print(df.shape)

Images: 31811
            image                                            caption
0  1000092795.jpg   Two young guys with shaggy hair look at their...
1  1000092795.jpg   Two young , White males are outside near many...
2  1000092795.jpg   Two men in green shirts are standing in a yard .
3  1000092795.jpg       A man in a blue shirt standing in a garden .
4  1000092795.jpg            Two friends enjoy time spent together .
Index(['image', 'caption'], dtype='object')
(158915, 2)


In [7]:
images = df["image"].unique().tolist()

print("Unique images:", len(images))

Unique images: 31783


In [9]:
# Create fixed split, 80, 10, 10
train_imgs, temp_imgs = train_test_split(
    images,
    train_size=25426,
    random_state=42,
    shuffle=True
)

val_imgs, test_imgs = train_test_split(
    temp_imgs,
    test_size=3178,
    random_state=42,
    shuffle=True
)

print(len(train_imgs))
print(len(val_imgs))
print(len(test_imgs))

25426
3179
3178


In [10]:
# Train test and val dataset
train_df = df[df["image"].isin(train_imgs)].reset_index(drop=True)

val_df = df[df["image"].isin(val_imgs)].reset_index(drop=True)

test_df = df[df["image"].isin(test_imgs)].reset_index(drop=True)

In [11]:
# Save files in drive
import pickle

split_dict = {
    "train": train_imgs,
    "val": val_imgs,
    "test": test_imgs
}

with open(
    f"{DATASET_DIR}/flickr30k_split.pkl",
    "wb"
) as f:
    pickle.dump(split_dict, f)

In [12]:
# Test Load
with open(
    f"{DATASET_DIR}/flickr30k_split.pkl",
    "rb"
) as f:
    split_dict = pickle.load(f)

In [13]:
train_df.head()

,image,caption
0,1000092795.jpg,Two young guys with shaggy hair look at their...
1,1000092795.jpg,"Two young , White males are outside near many..."
2,1000092795.jpg,Two men in green shirts are standing in a yard .
3,1000092795.jpg,A man in a blue shirt standing in a garden .
4,1000092795.jpg,Two friends enjoy time spent together .


Data Preprocessing

In [14]:
import re
from collections import Counter
import pickle


In [15]:
# Clean captions

def clean_caption(text):
    text = text.lower()

    # remove punctuation
    text = re.sub(r"[^a-z0-9\s]", "", text)

    # collapse spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [17]:
print(train_df["caption"].isna().sum())
print(val_df["caption"].isna().sum())
print(test_df["caption"].isna().sum())

1
0
0


In [18]:
train_df["caption"].apply(type).value_counts()

,count
caption,
<class 'str'>,127129
<class 'float'>,1


In [19]:
train_df = train_df.dropna(subset=["caption"]).reset_index(drop=True)
val_df = val_df.dropna(subset=["caption"]).reset_index(drop=True)
test_df = test_df.dropna(subset=["caption"]).reset_index(drop=True)

In [20]:
# Apply on the datasets

train_df["caption"] = train_df["caption"].apply(clean_caption)
val_df["caption"] = val_df["caption"].apply(clean_caption)
test_df["caption"] = test_df["caption"].apply(clean_caption)

In [21]:
train_df.head()

,image,caption
0,1000092795.jpg,two young guys with shaggy hair look at their ...
1,1000092795.jpg,two young white males are outside near many bu...
2,1000092795.jpg,two men in green shirts are standing in a yard
3,1000092795.jpg,a man in a blue shirt standing in a garden
4,1000092795.jpg,two friends enjoy time spent together


In [22]:
# Add the specialised tokens

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
SOS_TOKEN = "<SOS>"
EOS_TOKEN = "<EOS>"

In [23]:
# Train data vocab

counter = Counter()

for caption in train_df["caption"]:
    counter.update(caption.split())

In [24]:
print("Unique words:", len(counter))

Unique words: 18251


In [25]:
# Create Vocab

MIN_FREQ = 5

vocab = {
    PAD_TOKEN: 0,
    UNK_TOKEN: 1,
    SOS_TOKEN: 2,
    EOS_TOKEN: 3
}

for word, freq in counter.items():
    if freq >= MIN_FREQ:
        vocab[word] = len(vocab)

idx2word = {idx: word for word, idx in vocab.items()}

In [26]:
print("Vocabulary size:", len(vocab))

Vocabulary size: 6983


In [27]:
# Encoding

def encode_caption(text, vocab):

    tokens = text.split()

    encoded = [vocab["<SOS>"]]

    for token in tokens:
        encoded.append(
            vocab.get(token, vocab["<UNK>"])
        )

    encoded.append(vocab["<EOS>"])

    return encoded

In [28]:
# Set Max caption length for batching

lengths = []

for caption in train_df["caption"]:
    lengths.append(
        len(caption.split()) + 2
    )

print("Max length:", max(lengths))
print("Average length:", sum(lengths)/len(lengths))

Max length: 75
Average length: 14.264581645415287


In [29]:
MAX_LEN = 30

In [30]:
# Save and reload vocab

with open(
    f"{DATASET_DIR}/vocab.pkl",
    "wb"
) as f:
    pickle.dump(vocab, f)

In [31]:
with open(
    f"{DATASET_DIR}/vocab.pkl",
    "rb"
) as f:
    vocab = pickle.load(f)

Dataset Classes

In [32]:
from torchvision import transforms
from torch.utils.data import Dataset
from PIL import Image
import os
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader

In [33]:
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [34]:
from collections import defaultdict
import random

train_caption_map = defaultdict(list)
val_caption_map = defaultdict(list)
test_caption_map = defaultdict(list)

bad_img = "861608773_bdafd5c996.jpg"

train_caption_map.pop(bad_img, None)
val_caption_map.pop(bad_img, None)
test_caption_map.pop(bad_img, None)

for _, row in train_df.iterrows():
    train_caption_map[row["image"]].append(row["caption"])

for _, row in val_df.iterrows():
    val_caption_map[row["image"]].append(row["caption"])

for _, row in test_df.iterrows():
    test_caption_map[row["image"]].append(row["caption"])


class Flickr8KRetrievalDataset(Dataset):
    def __init__(self, caption_map, image_dir, vocab, transform=None, random_caption=True):
        self.image_names = sorted(caption_map.keys())
        self.caption_map = caption_map
        self.image_dir = image_dir
        self.vocab = vocab
        self.transform = transform
        self.random_caption = random_caption

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        image_name = self.image_names[idx]

        captions = self.caption_map[image_name]

        if self.random_caption:
            caption = random.choice(captions)
        else:
            caption = captions[0]      # fixed caption for eval

        try:
          image = Image.open(
              os.path.join(self.image_dir, image_name)
          ).convert("RGB")

        except Exception:
            return self.__getitem__(
                (idx + 1) % len(self)
            )

        if self.transform:
            image = self.transform(image)

        caption = torch.tensor(
            encode_caption(caption, self.vocab),
            dtype=torch.long
        )

        return image, caption

In [35]:
class Flickr8KAllCaptionEvalDataset(Dataset):

    def __init__(self, caption_map, image_dir, vocab, transform=None):

        self.samples = []
        self.image_dir = image_dir
        self.vocab = vocab
        self.transform = transform

        for image_name in sorted(caption_map.keys()):

            for caption in caption_map[image_name]:

                self.samples.append(
                    (image_name, caption)
                )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        image_name, caption = self.samples[idx]

        image = Image.open(
            os.path.join(self.image_dir, image_name)
        ).convert("RGB")

        if self.transform:
            image = self.transform(image)

        caption = torch.tensor(
            encode_caption(caption, self.vocab),
            dtype=torch.long
        )

        return image, torch.tensor(caption)

In [36]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):

    images = []
    captions = []
    lengths = []

    for image, caption in batch:
        images.append(image)
        captions.append(caption)
        lengths.append(len(caption))

    images = torch.stack(images)

    captions = pad_sequence(
        captions,
        batch_first=True,
        padding_value=vocab["<PAD>"]
    )

    lengths = torch.tensor(lengths)

    return images, captions, lengths

In [37]:
train_dataset = Flickr8KRetrievalDataset(
    train_caption_map,
    IMAGE_DIR,
    vocab,
    image_transform,
    random_caption=True
)

val_dataset = Flickr8KRetrievalDataset(
    val_caption_map,
    IMAGE_DIR,
    vocab,
    image_transform,
    random_caption=False
)

test_dataset = Flickr8KRetrievalDataset(
    test_caption_map,
    IMAGE_DIR,
    vocab,
    image_transform,
    random_caption=False
)

In [38]:
all_caption_test_dataset = Flickr8KAllCaptionEvalDataset(
    test_caption_map,
    IMAGE_DIR,
    vocab,
    image_transform
)

In [39]:
print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))

25426
3179
3178


In [40]:
# Dataloaders for clip style loading
BATCH_SIZE = 256

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0
)

In [41]:
all_caption_test_loader = DataLoader(
    all_caption_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0
)

In [42]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [43]:
device

'cuda'

In [44]:
print("device" in globals())
print("text_encoder" in globals())
print("train_loader" in globals())

True
False
True


In [45]:
image, caption = train_dataset[0]

print(image.shape)

print(caption)

torch.Size([3, 224, 224])
tensor([ 2, 31, 32, 17, 31, 33, 34, 30, 17, 31, 35,  3])


ResNet50 Image Encoder

In [46]:
EMBED_DIM = 256

In [47]:
import torch
import torch.nn as nn
import torchvision.models as models
import torch.nn.functional as F

In [48]:
# Encoder Image
class ImageEncoder(nn.Module):

    def __init__(self, embed_dim=512):

        super().__init__()

        backbone = models.resnet50(
            weights=models.ResNet50_Weights.IMAGENET1K_V2
        )

        for param in backbone.parameters():
          param.requires_grad = False

        # Fine tune last two ResNet stages
        for param in backbone.layer3.parameters():
            param.requires_grad = True

        for param in backbone.layer4.parameters():
            param.requires_grad = True

        self.backbone = nn.Sequential(
            *list(backbone.children())[:-1]
        )

        self.projection = nn.Sequential(
            nn.Linear(2048, 1024),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(1024, embed_dim)
        )

    def forward(self, images):

        features = self.backbone(images)

        features = features.squeeze(-1).squeeze(-1)

        embeddings = self.projection(features)

        embeddings = F.normalize(
            embeddings,
            p=2,
            dim=1
        )

        return embeddings

In [49]:
#Text Encoder
from torch.nn.utils.rnn import pack_padded_sequence

class TextEncoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_dim=300,
        hidden_dim=512,
        pad_idx=0
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=pad_idx
        )

        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            num_layers=2,
            dropout=0.3,
            bidirectional=True,
            batch_first=True
        )

        self.projection = nn.Sequential(
            nn.Linear(hidden_dim * 2, 1024),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(1024, 512)
        )

    def forward(
        self,
        captions,
        lengths
    ):

        embedded = self.embedding(captions)

        packed = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _, (hidden, _) = self.lstm(packed)

        hidden = torch.cat(
            [hidden[-2], hidden[-1]],
            dim=1
        )

        embeddings = self.projection(hidden)

        embeddings = F.normalize(
            embeddings,
            p=2,
            dim=1
        )

        return embeddings

In [50]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


In [51]:
encoder = ImageEncoder(
    embed_dim=512
).to(device)

images, captions, lengths = next(iter(train_loader))

images = images.to(device)

with torch.no_grad():
    image_embeddings = encoder(images)

print(image_embeddings.shape)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 229MB/s]


torch.Size([256, 512])


In [52]:
# text encoder installation and test
text_encoder = TextEncoder(
    vocab_size=len(vocab),
    embed_dim=300,
    hidden_dim=512,
    pad_idx=vocab["<PAD>"]
).to(device)

images, captions, lengths = next(iter(train_loader))

captions = captions.to(device)

with torch.no_grad():
    text_embeddings = text_encoder(
        captions,
        lengths
    )

print(text_embeddings.shape)

torch.Size([256, 512])


In [53]:
# compatibility verification
with torch.no_grad():
    image_embeddings = encoder(images.to(device))

    text_embeddings = text_encoder(
        captions.to(device),
        lengths
    )

print(image_embeddings.shape)
print(text_embeddings.shape)

torch.Size([256, 512])
torch.Size([256, 512])


In [54]:
# Normalize embedding
import torch.nn.functional as F

image_embeddings = F.normalize(image_embeddings, dim=1)
text_embeddings = F.normalize(text_embeddings, dim=1)

print(image_embeddings.norm(dim=1).mean())
print(text_embeddings.norm(dim=1).mean())

tensor(1., device='cuda:0')
tensor(1., device='cuda:0')


In [55]:
# Joint Model
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

class ResNetLSTMRetrieval(nn.Module):
    def __init__(self, image_encoder, text_encoder):
        super().__init__()

        self.image_encoder = image_encoder
        self.text_encoder = text_encoder

        # Learnable temperature parameter
        self.logit_scale = nn.Parameter(
            torch.ones([]) * np.log(1 / 0.07)
        )

    def forward(self, images, captions, lengths):
        image_emb = self.image_encoder(images)
        text_emb = self.text_encoder(captions, lengths)

        # Normalize embeddings
        image_emb = F.normalize(image_emb, p=2, dim=1)
        text_emb = F.normalize(text_emb, p=2, dim=1)

        return image_emb, text_emb



model = ResNetLSTMRetrieval(
    image_encoder=encoder,
    text_encoder=text_encoder
).to(device)

In [56]:
# CLIP-Style Contrastive Loss
def clip_contrastive_loss(image_emb, text_emb, logit_scale):
    # Similarity matrix: [B, B]
    logits = torch.matmul(image_emb, text_emb.T)
    logits = logits * logit_scale.exp()

    targets = torch.arange(
        image_emb.size(0),
        device=image_emb.device
    )

    loss_i2t = F.cross_entropy(logits, targets)
    loss_t2i = F.cross_entropy(logits.T, targets)

    loss = (loss_i2t + loss_t2i) / 2

    return loss, logits

In [57]:
# Forward pass test
images, captions, lengths = next(iter(train_loader))

images = images.to(device)
captions = captions.to(device)

with torch.no_grad():
    image_emb, text_emb = model(
        images,
        captions,
        lengths
    )

print(image_emb.shape)
print(text_emb.shape)

torch.Size([256, 512])
torch.Size([256, 512])


In [58]:
# Loss test
loss, logits = clip_contrastive_loss(
    image_emb,
    text_emb,
    model.logit_scale
)

print(loss.item())
print(logits.shape)

5.625430583953857
torch.Size([256, 256])


In [59]:
# Training
optimizer = torch.optim.AdamW(
    [
        {
            "params": model.image_encoder.backbone.parameters(),
            "lr": 1e-5
        },
        {
            "params": model.image_encoder.projection.parameters(),
            "lr": 3e-4
        },
        {
            "params": model.text_encoder.parameters(),
            "lr": 3e-4
        },
        {
            "params": [model.logit_scale],
            "lr": 1e-4
        }
    ],
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=50
)

In [60]:
# Add Validation Loop
def evaluate(model, dataloader, device):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for images, captions, lengths in dataloader:
            images = images.to(device)
            captions = captions.to(device)

            image_emb, text_emb = model(
                images,
                captions,
                lengths
            )

            loss, _ = clip_contrastive_loss(
                image_emb,
                text_emb,
                model.logit_scale
            )

            total_loss += loss.item()

    return total_loss / len(dataloader)

In [61]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()
del images, captions
gc.collect()
torch.cuda.empty_cache()

In [63]:
# training
NUM_EPOCHS = 30
best_val_loss = float("inf")
BEST_MODEL_PATH = "/content/drive/MyDrive/models/30k/best_retrieval_model.pth"

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0

    for batch_idx, (images, captions, lengths) in enumerate(train_loader):
        images = images.to(device)
        captions = captions.to(device)

        optimizer.zero_grad()

        image_emb, text_emb = model(images, captions, lengths)
        loss, _ = clip_contrastive_loss(
            image_emb,
            text_emb,
            model.logit_scale
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0
        )
        optimizer.step()

        running_loss += loss.item()

        if batch_idx % 20 == 0:
          print(
              f"Epoch {epoch+1} | "
              f"Batch {batch_idx}/{len(train_loader)} | "
              f"Loss: {loss.item():.4f}"
          )

    train_loss = running_loss / len(train_loader)
    val_loss = evaluate(model, val_loader, device)

    print(
        f"Epoch {epoch+1:02d} | "
        f"Train: {train_loss:.4f} | "
        f"Val: {val_loss:.4f}"
    )

    scheduler.step()

    # Save the best model based on validation loss
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "epoch": epoch,
                "val_loss": val_loss
            },
            BEST_MODEL_PATH
        )
        print("  ✓ Saved best model")

Epoch 1 | Batch 0/100 | Loss: 4.5371
Epoch 1 | Batch 20/100 | Loss: 4.2276
Epoch 1 | Batch 40/100 | Loss: 3.9528
Epoch 1 | Batch 60/100 | Loss: 3.9699
Epoch 1 | Batch 80/100 | Loss: 3.6888
Epoch 01 | Train: 4.0255 | Val: 3.6847
  ✓ Saved best model
Epoch 2 | Batch 0/100 | Loss: 3.6911
Epoch 2 | Batch 20/100 | Loss: 3.3873
Epoch 2 | Batch 40/100 | Loss: 3.3927
Epoch 2 | Batch 60/100 | Loss: 3.1579
Epoch 2 | Batch 80/100 | Loss: 3.1225
Epoch 02 | Train: 3.3515 | Val: 3.2905
  ✓ Saved best model
Epoch 3 | Batch 0/100 | Loss: 3.0535
Epoch 3 | Batch 20/100 | Loss: 2.9957
Epoch 3 | Batch 40/100 | Loss: 2.8300
Epoch 3 | Batch 60/100 | Loss: 2.7407
Epoch 3 | Batch 80/100 | Loss: 2.6582
Epoch 03 | Train: 2.9297 | Val: 3.0035
  ✓ Saved best model
Epoch 4 | Batch 0/100 | Loss: 2.5718
Epoch 4 | Batch 20/100 | Loss: 2.5726
Epoch 4 | Batch 40/100 | Loss: 2.6258
Epoch 4 | Batch 60/100 | Loss: 2.6341
Epoch 4 | Batch 80/100 | Loss: 2.5877
Epoch 04 | Train: 2.6219 | Val: 2.8174
  ✓ Saved best model
Epoc

In [64]:
FINAL_MODEL_PATH = "/content/drive/MyDrive/models/30k/best_retrieval_model.pth"

torch.save(model.state_dict(), BEST_MODEL_PATH)

Evaluation

In [65]:
def extract_embeddings(model, dataloader, device):
    model.load_state_dict(
    torch.load(FINAL_MODEL_PATH, map_location=device)
    )

    model.eval()


    image_embeddings = []
    text_embeddings = []

    with torch.no_grad():
        for images, captions, lengths in dataloader:
            images = images.to(device)
            captions = captions.to(device)

            img_emb, txt_emb = model(images, captions, lengths)

            image_embeddings.append(img_emb.cpu())
            text_embeddings.append(txt_emb.cpu())

    image_embeddings = torch.cat(image_embeddings, dim=0)
    text_embeddings = torch.cat(text_embeddings, dim=0)

    return image_embeddings, text_embeddings

In [66]:
image_embs, text_embs = extract_embeddings(
    model,
    all_caption_test_loader,
    device
)

print(image_embs.shape)
print(text_embs.shape)

/tmp/ipykernel_6146/340369310.py:37: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return image, torch.tensor(caption)


torch.Size([15890, 512])
torch.Size([15890, 512])


In [67]:
# Similarity Matrix
unique_image_embs = image_embs[::5]

similarity = unique_image_embs @ text_embs.T

print("Image embeddings:", unique_image_embs.shape)
print("Text embeddings:", text_embs.shape)
print("Similarity:", similarity.shape)

Image embeddings: torch.Size([3178, 512])
Text embeddings: torch.Size([15890, 512])
Similarity: torch.Size([3178, 15890])


In [68]:
# image -> text
def image_to_text_recall(similarity, k):

    correct = 0

    for img_idx in range(similarity.shape[0]):

        gt_caps = set(
            range(
                img_idx * 5,
                img_idx * 5 + 5
            )
        )

        topk = similarity[img_idx].topk(k).indices.tolist()

        if any(idx in gt_caps for idx in topk):
            correct += 1

    return correct / similarity.shape[0]


In [69]:
def text_to_image_recall(similarity, k):

    similarity_t = similarity.T

    correct = 0

    for cap_idx in range(similarity_t.shape[0]):

        gt_img = cap_idx // 5

        topk = similarity_t[cap_idx]\
            .topk(k)\
            .indices\
            .tolist()

        if gt_img in topk:
            correct += 1

    return correct / similarity_t.shape[0]


In [70]:
def image_to_text_mrr(similarity):

    reciprocal_ranks = []

    for img_idx in range(similarity.shape[0]):

        gt_caps = set(
            range(
                img_idx * 5,
                img_idx * 5 + 5
            )
        )

        sorted_idx = torch.argsort(
            similarity[img_idx],
            descending=True
        )

        best_rank = float("inf")

        for cap in gt_caps:

            rank = (
                (sorted_idx == cap)
                .nonzero(as_tuple=True)[0]
                .item()
            ) + 1

            best_rank = min(best_rank, rank)

        reciprocal_ranks.append(
            1.0 / best_rank
        )

    return np.mean(reciprocal_ranks)

In [71]:
def text_to_image_mrr(similarity):

    similarity_t2i = similarity.T

    reciprocal_ranks = []

    for cap_idx in range(similarity_t2i.shape[0]):

        gt_image = cap_idx // 5

        sorted_idx = torch.argsort(
            similarity_t2i[cap_idx],
            descending=True
        )

        rank = (
            (sorted_idx == gt_image)
            .nonzero(as_tuple=True)[0]
            .item()
        ) + 1

        reciprocal_ranks.append(
            1.0 / rank
        )

    return np.mean(reciprocal_ranks)

In [72]:
results = pd.DataFrame({
    "Metric": [
        "Recall@1",
        "Recall@5",
        "Recall@10",
        "MRR"
    ],
    "Image→Text": [
        image_to_text_recall(similarity, 1),
        image_to_text_recall(similarity, 5),
        image_to_text_recall(similarity, 10),
        image_to_text_mrr(similarity)
    ],
    "Text→Image": [
        text_to_image_recall(similarity, 1),
        text_to_image_recall(similarity, 5),
        text_to_image_recall(similarity, 10),
        text_to_image_mrr(similarity)
    ]
})

results["Image→Text"] = results["Image→Text"].round(6)
results["Text→Image"] = results["Text→Image"].round(6)

display(results)

,Metric,Image→Text,Text→Image
0,Recall@1,0.209251,0.154185
1,Recall@5,0.444934,0.374449
2,Recall@10,0.566394,0.491001
3,MRR,0.324694,0.262396
